#### Business Understanding
**Problem Statement:** Access to timely and reliable healthcare remains a major challenge, especially in regions with:
- Limited doctor-to-patient ratios 
- Long waiting times at clinics 
- High cost of consultations 
- Low health literacy among patients 
Many individuals:
- Ignore early symptoms 
- Misinterpret health information online 
- Delay seeking care until conditions worsen 
At the same time, general-purpose AI systems often:
- Provide unsafe or inaccurate medical advice 
- Lack grounding in trusted clinical guidelines 
- Do not assess urgency (triage) effectively 

This creates a critical gap: There is no widely accessible, safe, and intelligent system that can guide patients on what to do next based on their symptoms, without attempting full diagnosis or unsafe prescriptions.
Project Description:  Smart Doctor is a RAG-based AI Medical Triage & Advisory Assistant designed to:
- Interact with patients via text (and later voice) 
- Collect symptoms through structured conversation 
- Ask intelligent follow-up questions 
- Assess the urgency of the condition (triage) 
- Provide safe, evidence-based health advice 
- Recommend next steps (self-care, clinic visit, emergency care) 
- Store patient interaction data securely for future reference 

The system leverages:
- Large Language Models (LLMs) for conversation 
- Retrieval-Augmented Generation (RAG) for grounded medical knowledge 
- Rule-based logic for safe triage decision-making 

Importantly, the system:
Does NOT diagnose or prescribe medication, but instead supports decision-making and early intervention.
 Project Measurables (KPIs):
AI Performance Metrics

**1. Symptom Extraction Accuracy**
- Percentage of correctly identified symptoms from user input 
- Target: ≥ 85% accuracy 

**2. Triage Classification Accuracy**
- Agreement with medical guidelines or expert validation 
- Target: ≥ 90% for rule-based scenarios

**3. Response Grounding Score (RAG Quality)**
- Percentage of responses backed by retrieved medical sources 
- Target: ≥ 95% grounded responses 
 B. Safety Metrics

**4. Emergency Detection Recall**
- % of true emergencies correctly flagged 
- Target: ~100% (very critical) 

**5. Hallucination Rate**
- % of responses containing unsupported claims 
- Target: < 5% 

**6. Unsafe Recommendation Rate**

- Instances of: 
    - Prescriptions 
    - Confident diagnosis 
    - Target: 0% 
    
C. User Experience Metrics

**7. Conversation Completion Rate**
- Percentage of users who finish triage flow 
- Target: ≥ 80% 

**8. User Satisfaction Score**
- Feedback rating (1–5 scale) 
- Target: ≥ 4.0 

 D. System Metrics

**9. Response Time**
- Time taken to generate response 
- Target: < 3 seconds 

**10. System Reliability**
- Uptime / error rate 
- Target: ≥ 99% uptime 


In [3]:
# Import all important libraries
import os
import pandas as pd
import numpy as np
from bs4 import BeautifulSoup
import json
from urllib.parse import urljoin
import requests
from langchain.chat_models import init_chat_model
from langchain_openai import OpenAIEmbeddings
import faiss
from langchain_community.docstore.in_memory import InMemoryDocstore
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import PDFMinerLoader
from google import genai
from dotenv import load_dotenv
import time
load_dotenv()

gemini_api_key = os.getenv("GEMINI_API_KEY")
rag_api_key = os.getenv("RAG_API_KEY")



#### Load Documents

In [4]:
# Scrape Disease Fact Sheets from WHO website
BASE_URL = "https://www.who.int/"
URL = "https://www.who.int/news-room/fact-sheets"

# get response from the URL and parse it using BeautifulSoup
response = requests.get(URL)
soup = BeautifulSoup(response.text,'html.parser')
links = []

# # Extracting all the links of the disease fact sheets
for a in soup.find_all('a', href = True):
    href = a['href']
    if "/news-room/fact-sheets/detail/" in href:
        full_url = urljoin(BASE_URL, href)
        title = a.text.strip()
        links.append((title, full_url))

In [9]:
# Scrape the content of each disease fact sheet 
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"
}

def scrape_content(url):
    response = requests.get(url, headers = headers, timeout=10)
    soup = BeautifulSoup(response.text, 'html.parser')

    # Extract title
    title = soup.find('h1').text.strip() if soup.find('h1') else 'No title found'
    
    # Scrape the content of the page
    content = { }
    content_div = soup.find('article', class_= 'sf-detail-body-wrapper')

    # print(content_div.prettify())
    if content_div:
        # current_heading = None
        required_sections = [
                    'Overview',
                    'Signs and symptoms',
                    'Causes',
                    'Treatment and prevention',
                    'Self-care',
                    'WHO response'
                ]
        for tag in content_div.find_all(['h2', 'p','li']):
            # Content headings
            if tag.name == 'h2':
                heading = tag.get_text(strip=True)
                if heading in required_sections:
                    current_heading = heading
                    content[current_heading] = []
                else:
                    current_heading = None
            # Content paragraphs
            elif tag.name == 'p':
                text = tag.get_text(" ",strip=True)
                if text and current_heading:
                    content[current_heading].append(text)
            # Content list items
            elif tag.name == 'li':
                text = tag.get_text(" ",strip=True)
                if text and current_heading:
                    content[current_heading].append(f" - {text}")

    data = {
        "title": title,
        "content": content
    }

    return data

scraped_content = scrape_content(links[7][1])
print(scraped_content)

{'title': 'Anaemia', 'content': {'Overview': ['Anaemia is a condition in which the number of red blood cells or the haemoglobin concentration within them is lower than normal. It mainly affects women and children.', 'Anaemia occurs when there isn’t enough haemoglobin in the body to carry oxygen to the organs and tissues.', 'In severe cases, anaemia can cause poor cognitive and motor development in children. It can also cause problems for pregnant women and their babies.', 'Anaemia can be caused by poor nutrition, infections, chronic diseases, heavy menstruation, pregnancy issues and family history. It is often caused by a lack of iron in the blood.', 'Anaemia is preventable and treatable.', 'In many low- and lower-middle income settings, the most commonly- recognized causes of anaemia are iron deficiency and malaria.'], 'Signs and symptoms': ['Common and non-specific symptoms of anaemia include:', ' - tiredness', ' - dizziness or feeling light-headed', ' - cold hands and feet', ' - hea

#### Save Scraped Data in A CSV 

[{'title': 'Abortion', 'content': ''}]
[{'title': 'Abortion', 'content': ''}, {'title': 'Abuse of older people', 'content': ''}]
[{'title': 'Abortion', 'content': ''}, {'title': 'Abuse of older people', 'content': ''}, {'title': 'Adolescent and young adult health', 'content': ''}]
[{'title': 'Abortion', 'content': ''}, {'title': 'Abuse of older people', 'content': ''}, {'title': 'Adolescent and young adult health', 'content': ''}, {'title': 'Adolescent pregnancy', 'content': ''}]
[{'title': 'Abortion', 'content': ''}, {'title': 'Abuse of older people', 'content': ''}, {'title': 'Adolescent and young adult health', 'content': ''}, {'title': 'Adolescent pregnancy', 'content': ''}, {'title': 'Ageing and health', 'content': ''}]


#### Define GEMINI Model

In [5]:
# Create LLM client
client = genai.Client(api_key=gemini_api_key)
# Define the model name
model_name = "gemini-2.5-flash"
# Create a class for the Gemini model
class GEMINI_MODEL:
    # Initialize the class with the client and model name
    def __init__(self, client, model_name):
        self.client = client
        self.model_name = model_name
# Define a method to generate content using the Gemini model
    def generate_content(self, contents):
        response = self.client.models.generate_content(
            model = self.model_name,
            contents = contents
        )
        return response

model = GEMINI_MODEL(client, model_name)

#### Create Embeddings

In [6]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vector_store = FAISS(embeddings.embed_query, InMemoryDocstore({}), {})

OpenAIError: Missing credentials. Please pass an `api_key`, `workload_identity`, `admin_api_key`, or set the `OPENAI_API_KEY` or `OPENAI_ADMIN_KEY` environment variable.